In [8]:
import pandas as pd
import numpy as np

In [9]:
df=pd.read_parquet('data/hourly_demand_features.parquet')

In [10]:
df

,hour,PULocationID,demand,hour_of_day,cluster
0,2023-01-01 00:00:00,1,0.0,0,0
1,2023-01-01 00:00:00,2,0.0,0,0
2,2023-01-01 00:00:00,3,0.0,0,0
3,2023-01-01 00:00:00,4,19.0,0,2
4,2023-01-01 00:00:00,5,0.0,0,0
...,...,...,...,...,...
2303875,2023-12-31 23:00:00,261,19.0,23,2
2303876,2023-12-31 23:00:00,262,54.0,23,3
2303877,2023-12-31 23:00:00,263,97.0,23,1
2303878,2023-12-31 23:00:00,264,14.0,23,4


In [11]:
def add_lag_features(df):
    df = df.sort_values(["PULocationID", "hour"]).reset_index(drop=True)

    df["lag_1"] = df.groupby("PULocationID")["demand"].shift(1)
    df["lag_24"] = df.groupby("PULocationID")["demand"].shift(24)
    df["lag_168"] = df.groupby("PULocationID")["demand"].shift(168)

    return df

In [12]:
df_lagged = add_lag_features(df)

In [13]:
df_lagged.head()

,hour,PULocationID,demand,hour_of_day,cluster,lag_1,lag_24,lag_168
0,2023-01-01 00:00:00,1,0.0,0,0,NaN,NaN,NaN
1,2023-01-01 01:00:00,1,0.0,1,0,0.0,NaN,NaN
2,2023-01-01 02:00:00,1,0.0,2,0,0.0,NaN,NaN
3,2023-01-01 03:00:00,1,0.0,3,0,0.0,NaN,NaN
4,2023-01-01 04:00:00,1,0.0,4,0,0.0,NaN,NaN


In [14]:
df_lagged.dropna(inplace=True)

In [15]:
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

def build_features(df):
    """
    Input df must contain:
    - hour (datetime)
    - PULocationID
    - demand
    - lag_1, lag_24, lag_168
    - cluster_id
    """

    # ---------- BASIC TIME FEATURES ----------
    df["hour_of_day"] = df["hour"].dt.hour
    df["day_of_week"] = df["hour"].dt.dayofweek
    df["day_of_month"] = df["hour"].dt.day
    df["month"] = df["hour"].dt.month

    # ---------- FLAGS ----------
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["is_peak_hour"] = df["hour_of_day"].isin([7, 8, 9, 16, 17, 18]).astype(int)

    # ---------- CYCLICAL ENCODING ----------
    df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # ---------- HOLIDAY FEATURE ----------
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(
        start=df["hour"].min(),
        end=df["hour"].max()
    )

    df["is_holiday"] = df["hour"].dt.normalize().isin(holidays).astype(int)

    # ---------- LOG TRANSFORM (CRITICAL) ----------
    for col in ["demand", "lag_1", "lag_24", "lag_168"]:
        df[col] = np.log1p(df[col])

    # ---------- DROP RAW TIMESTAMP ----------
    #df = df.drop(columns=["hour"])

    return df


In [16]:
df_final = build_features(df_lagged)

In [27]:
df_final

,hour,PULocationID,demand,hour_of_day,cluster,lag_1,lag_24,lag_168,day_of_week,day_of_month,month,is_weekend,is_peak_hour,hour_sin,hour_cos,dow_sin,dow_cos,is_holiday
168,2023-01-08 00:00:00,1,0.000000,0,0,0.000000,0.000000,0.000000,6,8,1,1,0,0.000000,1.000000,-0.781831,0.62349,0
169,2023-01-08 01:00:00,1,0.000000,1,0,0.000000,0.000000,0.000000,6,8,1,1,0,0.258819,0.965926,-0.781831,0.62349,0
170,2023-01-08 02:00:00,1,0.000000,2,0,0.000000,0.000000,0.000000,6,8,1,1,0,0.500000,0.866025,-0.781831,0.62349,0
171,2023-01-08 03:00:00,1,0.000000,3,0,0.000000,0.000000,0.000000,6,8,1,1,0,0.707107,0.707107,-0.781831,0.62349,0
172,2023-01-08 04:00:00,1,0.000000,4,0,0.000000,0.000000,0.000000,6,8,1,1,0,0.866025,0.500000,-0.781831,0.62349,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2303875,2023-12-31 19:00:00,265,0.000000,19,2,1.609438,0.693147,1.791759,6,31,12,1,0,-0.965926,0.258819,-0.781831,0.62349,0
2303876,2023-12-31 20:00:00,265,0.000000,20,2,0.000000,1.386294,1.945910,6,31,12,1,0,-0.866025,0.500000,-0.781831,0.62349,0
2303877,2023-12-31 21:00:00,265,1.386294,21,2,0.000000,1.791759,1.098612,6,31,12,1,0,-0.707107,0.707107,-0.781831,0.62349,0
2303878,2023-12-31 22:00:00,265,1.386294,22,2,1.386294,1.609438,1.386294,6,31,12,1,0,-0.500000,0.866025,-0.781831,0.62349,0


from meteostat import Point, Hourly
from datetime import datetime
import pandas as pd


In [18]:
import pandas as pd
import requests


In [19]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "snowfall",
        "wind_speed_10m"
    ],
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data = response.json()


In [20]:
weather = pd.DataFrame({
    "hour": pd.to_datetime(data["hourly"]["time"]),
    "temp": data["hourly"]["temperature_2m"],
    "rhum": data["hourly"]["relative_humidity_2m"],
    "prcp": data["hourly"]["precipitation"],
    "snow": data["hourly"]["snowfall"],
    "wspd": data["hourly"]["wind_speed_10m"]
})


In [21]:
weather.head()
weather.shape


(8760, 6)

In [22]:
weather.head()

,hour,temp,rhum,prcp,snow,wspd
0,2023-01-01 00:00:00,10.6,99,1.0,0.0,7.4
1,2023-01-01 01:00:00,10.6,98,0.1,0.0,13.1
2,2023-01-01 02:00:00,10.4,96,0.0,0.0,16.0
3,2023-01-01 03:00:00,9.8,95,0.0,0.0,16.0
4,2023-01-01 04:00:00,9.1,95,0.0,0.0,15.8


In [23]:
weather = weather.fillna(0)


In [28]:
df = df_final.merge(weather, on="hour", how="left")


In [29]:
df.head()

,hour,PULocationID,demand,hour_of_day,cluster,lag_1,lag_24,lag_168,day_of_week,day_of_month,...,hour_sin,hour_cos,dow_sin,dow_cos,is_holiday,temp,rhum,prcp,snow,wspd
0,2023-01-08 00:00:00,1,0.0,0,0,0.0,0.0,0.0,6,8,...,0.000000,1.000000,-0.781831,0.62349,0,1.5,73,0.0,0.0,11.6
1,2023-01-08 01:00:00,1,0.0,1,0,0.0,0.0,0.0,6,8,...,0.258819,0.965926,-0.781831,0.62349,0,1.1,71,0.0,0.0,11.2
2,2023-01-08 02:00:00,1,0.0,2,0,0.0,0.0,0.0,6,8,...,0.500000,0.866025,-0.781831,0.62349,0,0.8,70,0.0,0.0,13.7
3,2023-01-08 03:00:00,1,0.0,3,0,0.0,0.0,0.0,6,8,...,0.707107,0.707107,-0.781831,0.62349,0,0.2,72,0.0,0.0,13.6
4,2023-01-08 04:00:00,1,0.0,4,0,0.0,0.0,0.0,6,8,...,0.866025,0.500000,-0.781831,0.62349,0,-0.4,72,0.0,0.0,11.3


In [30]:
df["is_rain"] = (df["prcp"] > 0).astype(int)
df["is_snow"] = (df["snow"] > 0).astype(int)
df["is_cold"] = (df["temp"] < 0).astype(int)
df["is_hot"]  = (df["temp"] > 30).astype(int)
df.head()

,hour,PULocationID,demand,hour_of_day,cluster,lag_1,lag_24,lag_168,day_of_week,day_of_month,...,is_holiday,temp,rhum,prcp,snow,wspd,is_rain,is_snow,is_cold,is_hot
0,2023-01-08 00:00:00,1,0.0,0,0,0.0,0.0,0.0,6,8,...,0,1.5,73,0.0,0.0,11.6,0,0,0,0
1,2023-01-08 01:00:00,1,0.0,1,0,0.0,0.0,0.0,6,8,...,0,1.1,71,0.0,0.0,11.2,0,0,0,0
2,2023-01-08 02:00:00,1,0.0,2,0,0.0,0.0,0.0,6,8,...,0,0.8,70,0.0,0.0,13.7,0,0,0,0
3,2023-01-08 03:00:00,1,0.0,3,0,0.0,0.0,0.0,6,8,...,0,0.2,72,0.0,0.0,13.6,0,0,0,0
4,2023-01-08 04:00:00,1,0.0,4,0,0.0,0.0,0.0,6,8,...,0,-0.4,72,0.0,0.0,11.3,0,0,1,0


In [31]:
df.shape

(2259696, 27)

In [32]:
df.to_parquet(
    "data/ml_features_with_weather.parquet",
    index=False
)
